In [0]:
%run ../silver/00_silver_helpers

In [0]:
df= read_table('products')
display(df)

In [0]:
logger = get_logger("silver_products")

try:

    logger.info("Starting Silver Products transformation")

    # ---------------------------------------------------------
    # Data Type Conversion
    # ---------------------------------------------------------

    logger.info("Updating data types of the columns")
    logger.info("Trimming leading/trailing spaces from string columns")

    df = df.select(

        trim(col("product_id").try_cast("string")).alias("product_id"),

        trim(col("product_name").try_cast("string")).alias("product_name"),

        trim(col("category").try_cast("string")).alias("category"),

        trim(col("subcategory").try_cast("string")).alias("subcategory"),

        trim(col("brand").try_cast("string")).alias("brand"),

        col("unit_cost").try_cast("decimal(10,2)").alias("unit_cost"),

        col("_ingestion_timestamp"),

        col("_source_file")

    )

    logger.info("Data type conversion and string trimming completed")

    # ---------------------------------------------------------
    # Duplicate Product IDs
    # ---------------------------------------------------------

    logger.info("Checking for duplicate product_id values")

    before_count = df.count()

    distinct_count = df.dropDuplicates(
        ["product_id"]
    ).count()

    duplicate_count = before_count - distinct_count

    logger.info(
        f"Duplicate product_id records found: {duplicate_count}"
    )

    df = df.dropDuplicates(["product_id"])

    logger.info("Duplicate product_id records removed")

    # ---------------------------------------------------------
    # Null Product IDs
    # ---------------------------------------------------------

    logger.info("Checking for null product_id values")

    before_count = df.count()

    df = df.where(
        col("product_id").isNotNull()
    )

    after_count = df.count()

    records_dropped = before_count - after_count

    logger.info(
        f"Records dropped due to null product_id: {records_dropped}"
    )

    # ---------------------------------------------------------
    # Unit Cost Validation
    # ---------------------------------------------------------

    logger.info(
        "Checking for null or negative unit_cost values"
    )

    invalid_unit_cost_count = df.where(
        col("unit_cost").isNull() |
        (col("unit_cost") < 0)
    ).count()

    logger.info(
        f"Invalid unit_cost records found: {invalid_unit_cost_count}"
    )

    if invalid_unit_cost_count > 0:

        logger.warning(
            "Null/negative unit_cost values detected"
        )

        logger.info(
            "Replacing null and negative unit_cost values with 0"
        )

        df = df.withColumn(
            "unit_cost",
            when(
                col("unit_cost").isNull() |
                (col("unit_cost") < 0),
                lit(0).cast("decimal(10,2)")
            ).otherwise(
                col("unit_cost")
            )
        )

        logger.warning(
            "Invalid unit_cost values replaced with 0"
        )

        logger.warning(
            "Pricing/Product team should investigate the upstream data quality issue"
        )

    else:

        logger.info(
            "No null or negative unit_cost values found"
        )

    df.printSchema()

    display(df)
    # ---------------------------------------------------------
    # Schema
    # ---------------------------------------------------------

    logger.info(
        f"Creating schema if it does not exist: "
        f"{catalog_name}.{schema_name}"
    )

    spark.sql(
        f"""
        CREATE SCHEMA IF NOT EXISTS
        {catalog_name}.{schema_name}
        """
    )

    logger.info(
        f"Schema ready: {catalog_name}.{schema_name}"
    )

    # ---------------------------------------------------------
    # Save
    # ---------------------------------------------------------

    logger.info("Saving Silver Products table")

    save_table(
        df,
        "products_clean"
    )

    logger.info(
        "Silver Products table saved successfully"
    )

    logger.info(
        "Silver Products transformation completed successfully"
    )


except Exception:

    logger.exception(
        "Silver Products transformation failed"
    )

    raise